<a href="https://colab.research.google.com/github/CaroleSchoepfer5/BugNet_SoilArthro/blob/main/BugNet_SoilArthro_FlatBug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install rawpy tqdm pillow

# Clone the flat-bug repo
!git clone https://github.com/darsa-group/flat-bug.git --branch main --single-branch flat-bug

# --- Small compatibility patch (because Colab often uses Python 3.10) ---

import re
import os

def find_and_replace_in_file(file_path, search_pattern, replacement_text):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        updated_content = re.sub(search_pattern, replacement_text, content)
        with open(file_path, 'w', encoding='utf-8') as file:
            file.write(updated_content)
        print(f"Replaced text in '{file_path}' successfully.")
    except FileNotFoundError:
        print(f"File '{file_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Allow Python 3.10
find_and_replace_in_file(
    'flat-bug/pyproject.toml',
    r'requires-python = ">=3.11"',
    'requires-python = ">=3.10"'
)

# Add typing_extensions.Self where needed
self_replace = "from typing_extensions import Self"
find_and_replace_in_file(
    'flat-bug/src/flat_bug/predictor.py',
    r'from typing import Any, List, Optional, Self, Tuple, Union',
    f'from typing import Any, List, Optional, Tuple, Union\n{self_replace}'
)
find_and_replace_in_file(
    'flat-bug/src/flat_bug/augmentations.py',
    r'from typing import Dict, List, Optional, Self, Tuple, Union',
    f'from typing import Dict, List, Optional, Tuple, Union\n{self_replace}'
)
find_and_replace_in_file(
    'flat-bug/src/flat_bug/trainers.py',
    r'from typing import Any, Dict, List, Optional, Self, Tuple, Union',
    f'from typing import Any, Dict, List, Optional, Tuple, Union\n{self_replace}'
)
find_and_replace_in_file(
    'flat-bug/src/flat_bug/datasets.py',
    r'from typing import Dict, List, Optional, Self, Tuple, Union',
    f'from typing import Dict, List, Optional, Tuple, Union\n{self_replace}'
)

# Install flat-bug package in editable mode
!pip install -e flat-bug

# Make sure Python can find it
import sys
sys.path.append("/content/flat-bug/src")

print("Setup finished.")


In [4]:
import os
import glob
import io
import uuid
import re
import shutil
import time
from typing import List, Tuple, Union, Optional

from tqdm import tqdm
import numpy as np
import torch
import rawpy
from PIL import Image

from flat_bug.predictor import Predictor, TensorPredictions


# =============================================================================
# Helper functions
# =============================================================================

def parse_image(
    images: Optional[
        Union[
            np.ndarray,
            bytes,
            str,
            Union[List[Union[np.ndarray, bytes, str]], Tuple[Union[np.ndarray, bytes, str]]],
        ]
    ],
    device: Union[torch.device, str] = "cpu",
):
    """
    Load an image and convert it to a PyTorch tensor.

    Accepted input formats:
    - file path as string (.jpg, .jpeg, .png, .dng)
    - image as bytes
    - image as NumPy array
    - list/tuple of any of the above
    """

    # If several images are provided, parse each one separately
    if isinstance(images, (list, tuple)):
        return [parse_image(image, device) for image in images]

    # If a file path is provided, open the image from disk
    elif isinstance(images, str):

        # DNG/RAW images require rawpy
        if re.search(re.compile(r"\.dng$", re.IGNORECASE), images):
            with rawpy.imread(images) as raw:
                images = raw.postprocess()
                images = Image.fromarray(images)

        # Standard image formats can be opened directly with PIL
        else:
            images = Image.open(images)

    # If image bytes are provided, open them from memory
    elif isinstance(images, bytes):
        images = Image.open(io.BytesIO(images))

    # NumPy arrays can be used directly
    elif isinstance(images, np.ndarray):
        pass

    else:
        raise ValueError(
            f"Expected image(s) to be a np.ndarray, string, bytes, "
            f"or list/tuple of these, but got {type(images)}"
        )

    # Convert image to NumPy array and then to PyTorch tensor.
    # The dimension order is changed from (height, width, channels)
    # to (channels, height, width), which is expected by the model.
    image = np.array(images)
    return torch.from_numpy(image).permute(2, 0, 1).to(device)


def generate_uuid() -> str:
    """
    Generate a short unique identifier for temporary output folders/files.
    """
    return str(uuid.uuid4())[::3]


def wait_until_crops_finished(folder, identifier, timeout=60, stable_time=1.2):
    """
    Wait until FlatBug has finished writing crop files.

    FlatBug saves crops asynchronously. Without this waiting step, the script may
    continue before all crops are written, resulting in missing crops.

    The function checks the number of crop files repeatedly and continues only
    once the number of files has stopped changing for `stable_time` seconds.
    Be aware that this step will slow the processing but ensure safe storing of crops on drive
    """

    pattern = os.path.join(folder, f"crop*{identifier}.png")

    start = time.time()
    last_count = -1
    last_change = time.time()

    while time.time() - start < timeout:
        files = glob.glob(pattern)
        count = len(files)

        # If new files appeared, reset the stability timer
        if count != last_count:
            last_count = count
            last_change = time.time()

        # Continue once the file count has remained stable long enough
        if time.time() - last_change >= stable_time:
            return sorted(files)

        time.sleep(0.2)

    # If the timeout is reached, return whatever has been saved so far
    return sorted(glob.glob(pattern))


# =============================================================================
# FlatBug model wrapper
# =============================================================================

class Localizer(Predictor):
    """
    Wrapper around the FlatBug Predictor.

    This class runs the FlatBug model on one or several images, saves the detected
    organisms as crop images, and stores the paths to these crops.
    """

    def predict(
        self,
        images: Optional[
            Union[
                np.ndarray,
                bytes,
                str,
                Union[List[Union[np.ndarray, bytes, str]], Tuple[Union[np.ndarray, bytes, str]]],
            ]
        ],
        do_plot: bool | List[bool] = False,
        include_crops: bool = False,
        outdir: str = "output",
    ) -> dict:
        """
        Run FlatBug on one or more images.

        Returns a dictionary containing:
        - uuids: unique identifiers for each processed image
        - predictions: model prediction metadata
        - crops: paths to saved crop images
        - visualizations: optional prediction visualization paths
        """

        data = {
            "uuids": [],
            "predictions": [],
            "crops": [],
            "visualizations": [],
        }

        # Convert single image input to a list for consistent processing
        if not isinstance(images, (list, tuple)):
            images = [images]

        # Convert do_plot to a list matching the number of images
        if not isinstance(do_plot, list):
            if isinstance(do_plot, tuple):
                do_plot = list(do_plot)
            else:
                do_plot = [do_plot]

            if len(do_plot) == 1 and len(images) > 1:
                do_plot = do_plot * len(images)

        if not all(isinstance(plot, bool) for plot in do_plot):
            raise ValueError(
                f"Expected do_plot to be a boolean or list of booleans, but got {do_plot}"
            )

        # Process images one by one.
        # tqdm shows one updating progress bar instead of manual print statements.
        for i, image in enumerate(
            tqdm(
                images,
                desc="Localizing insects",
                unit="image",
                leave=True,
                dynamic_ncols=True,
            )
        ):

            # Use the original image name as basis for the crop names
            if isinstance(image, str):
                image_identifier = os.path.splitext(os.path.basename(image))[0]
            else:
                image_identifier = "DUMMY"

            # Load image and convert it to model input format
            image_tensor = parse_image(image, self._device)

            # Create unique temporary folder for this image
            identifier = generate_uuid()
            this_outdir = os.path.join(outdir, identifier)
            os.makedirs(this_outdir, exist_ok=True)

            # Run FlatBug prediction
            predictions: TensorPredictions = self.pyramid_predictions(
                image_tensor,
                "DUMMY_PATH_STR",
                scale_before=1,
            )

            # Save detected organisms as crop images.
            # Crop filenames include the original image name and unique identifier.
            predictions.save_crops(
                outdir=this_outdir,
                basename=image_identifier,
                mask=True,
                identifier=identifier,
            )

            # Wait until all asynchronously saved crops are actually written.
            # This prevents missing crops.
            crops = wait_until_crops_finished(this_outdir, identifier)

            # Optional visualization of predictions
            if do_plot[i]:
                visualization_dir = os.path.join(
                    os.path.dirname(this_outdir),
                    "visualization",
                )
                os.makedirs(visualization_dir, exist_ok=True)

                predict_image = os.path.join(
                    visualization_dir,
                    f"{identifier}_visualization.jpg",
                )

                predictions.plot(outpath=predict_image, scale=1 / 2)

            else:
                predict_image = None

            # Store results for this image
            data["uuids"].append(identifier)
            data["visualizations"].append(predict_image)
            data["crops"].append(crops)
            data["predictions"].append(predictions.json_data)

        return data


# =============================================================================
# Model/device setup
# =============================================================================

def get_defaults():
    """
    Select GPU if available, otherwise CPU.

    torch.float16 is faster and uses less memory on compatible GPUs.
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16
    return device, dtype


# =============================================================================
# Main batch-processing function
# =============================================================================

def batch_flatbug(
    input_folder: str,
    output_folder: str,
    weights_path: str,
    score_threshold: float = 0.25,
):
    """
    Run FlatBug on all images in a folder.

    The final crops are saved in `output_folder` with names:
    OriginalImageName_1.png
    OriginalImageName_2.png
    ...

    Parameters
    ----------
    input_folder:
        Folder containing input images.

    output_folder:
        Folder where final crop images will be saved.

    weights_path:
        Path to the fine-tuned FlatBug model weights (.pt file).

    score_threshold:
        Detection confidence threshold. Lower values detect more candidates,
        but also increase false positives.
    """

    # -------------------------------------------------------------------------
    # 1) Collect all input images
    # -------------------------------------------------------------------------

    image_paths = []

    for root, dirs, files in os.walk(input_folder):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".dng")):
                image_paths.append(os.path.join(root, f))

    image_paths = sorted(image_paths)

    if not image_paths:
        raise ValueError(f"No images found in {input_folder}")

    print(f"Found {len(image_paths)} images.")

    # -------------------------------------------------------------------------
    # 2) Prepare output folders
    # -------------------------------------------------------------------------

    os.makedirs(output_folder, exist_ok=True)

    # Temporary folder used by FlatBug before crops are renamed and collected
    tmp_outdir = os.path.join(output_folder, "_tmp_flatbug")
    os.makedirs(tmp_outdir, exist_ok=True)

    # -------------------------------------------------------------------------
    # 3) Set up model
    # -------------------------------------------------------------------------

    device, dtype = get_defaults()
    print(f"Using device: {device}, dtype: {dtype}")

    model = Localizer(model=weights_path, device=device, dtype=dtype)

    model.set_hyperparameters(
        SCORE_THRESHOLD=score_threshold,
        EDGE_CASE_MARGIN=32,
        MIN_MAX_OBJ_SIZE=(16, 768),
        TIME=False,
    )

    # -------------------------------------------------------------------------
    # 4) Run FlatBug predictions
    # -------------------------------------------------------------------------

    results = model.predict(
        images=image_paths,
        do_plot=False,
        include_crops=False,
        outdir=tmp_outdir,
    )

    # -------------------------------------------------------------------------
    # 5) Move crops into final output folder and rename them
    # -------------------------------------------------------------------------

    total_crops = 0

    for img_path, crop_paths in zip(image_paths, results["crops"]):

        # Original image name without file extension
        base = os.path.splitext(os.path.basename(img_path))[0]

        for idx, crop_path in enumerate(crop_paths, start=1):

            # Skip missing files, just in case
            if not os.path.isfile(crop_path):
                continue

            # Final crop name keeps the source image name
            new_name = f"{base}_{idx}.png"
            new_path = os.path.join(output_folder, new_name)

            shutil.move(crop_path, new_path)
            total_crops += 1

    # -------------------------------------------------------------------------
    # 6) Clean up temporary files
    # -------------------------------------------------------------------------

    shutil.rmtree(tmp_outdir, ignore_errors=True)

    print(f"Processed {len(image_paths)} images.")
    print(f"Saved {total_crops} crops to: {output_folder}")

In [ ]:
# >>> EDIT THESE THREE LINES TO MATCH YOUR DRIVE <<<

input_folder = "/content/drive/MyDrive/FlatBug/input"   # where you put your test images
output_folder = "/content/drive/MyDrive/FlatBug/output" # where you want crops to go
weights_path = "/content/drive/MyDrive/FlatBug/models/flatbug_weights.pt"  # your .pt file

batch_flatbug(
    input_folder=input_folder,
    output_folder=output_folder,
    weights_path=weights_path,
    score_threshold=0.22,   # you can tweak this later
)